# Informe de Trabajo: Simulación de Eventos Discretos

## Planteamiento del Problema

### Escenario: Aeropuerto de Barajas

En el Aeropuerto de Barajas, se desea conocer cuánto tiempo permanecen vacías las pistas de aterrizaje. Se sabe que el aeropuerto dispone de un máximo de 5 pistas dedicadas a aviones de carga, y se considera que una pista está ocupada cuando un avión se encuentra aterrizando, despegando, o bien cuando está cargando o descargando mercancía, o durante el embarque o desembarque de pasajeros.

Se conoce que el tiempo entre llegadas de los aviones sigue una distribución exponencial con λ = 20 minutos. Si un avión llega al aeropuerto y no hay pistas libres, permanece en espera hasta que alguna se desocupe (en caso de que haya varios aviones en esta situación, se forma una cola para el aterrizaje).

Además, el tiempo de carga y descarga de cada avión sigue una distribución exponencial con λ = 30 minutos. El tiempo de aterrizaje y despegue se modela con una distribución normal N(10, 5), y la probabilidad de que un avión realice carga y/o descarga en cada viaje sigue una distribución uniforme.

Por otro lado, se sabe que los aviones tienen una probabilidad de 0.1 de sufrir una avería. Cuando esto ocurre, el avión debe ser reparado en un tiempo que sigue una distribución exponencial con λ = 15 minutos. Estas averías se detectan justo antes del despegue.

Igualmente, durante su estancia en pista, cada avión debe repostar combustible. El tiempo de recarga sigue una distribución exponencial con λ = 30 minutos y comienza en el mismo momento en que el avión aterriza.

Se asume, además, que los aviones pueden aterrizar en cualquier pista sin ningún tipo de preferencia o restricción.

El objetivo es simular el comportamiento del aeropuerto durante una semana (T = 10080 minutos) para estimar el tiempo total que cada una de las pistas permanece vacía.



## Enfoque de la Solución

Para abordar este problema, hemos seguido un planteamiento estructurado y realista, basado en los siguientes pilares:

###  Modelo de colas con servidores paralelos

Hemos representado el aeropuerto como un sistema de colas M/G/5 con disciplina FIFO (primero en llegar, primero en ser atendido). En este modelo, los aviones son los clientes y las 5 pistas son los servidores, que operan en paralelo y son intercambiables entre sí. Esta abstracción nos permite estudiar el comportamiento del sistema con herramientas de simulación de eventos discretos.

###  Diseño modular del código

Para garantizar que el código sea claro, fácil de mantener y reutilizable, hemos dividido la solución en cuatro módulos independientes:

- **`generadores.py`**: Contiene las funciones encargadas de generar variables aleatorias continuas utilizando métodos estocásticos implementados manualmente.
- **`modelo_barajas.py`**: Define las reglas específicas del problema, como los tiempos de llegada y la secuencia de operaciones que componen el tiempo de servicio de cada avión.
- **`motor_simulacion.py`**: Implementa un motor genérico de simulación de eventos discretos basado en una cola de prioridad (min-heap), que permite gestionar los eventos de manera eficiente.
- **`main.py`**: Es el módulo principal, encargado de lanzar múltiples réplicas de la simulación y detener el proceso cuando se alcanza la precisión deseada.

###  Asignación uniforme de pistas

Para evitar que algunas pistas se saturaran artificialmente por un sesgo en la selección (por ejemplo, que siempre se eligiera la primera pista disponible), hemos implementado un mecanismo que asigna aleatoriamente una pista libre con probabilidad uniforme. De esta forma, la carga de trabajo se distribuye de manera equitativa entre todas las pistas.

### 4. Control del tiempo de inactividad hasta el horizonte T

Llevamos un registro detallado de los intervalos en los que cada pista permanece vacía. Si una pista queda inactiva antes o en el instante T = 10.080 minutos, su tiempo de inactividad se contabiliza únicamente hasta ese límite, ignorando cualquier evento que ocurra después del cierre de la simulación. Esto nos permite obtener estimaciones coherentes con el período de estudio.

---

## Modelo de Simulación Implementado

### Generación de variables aleatorias

Para simular la aleatoriedad del sistema, hemos implementado manualmente dos métodos clásicos de generación de variables:

* **Distribución exponencial (método de la transformada inversa):**

$$X = -\frac{1}{\lambda} \ln(1 - U), \quad U \sim \text{Uniforme}(0, 1)$$

Donde $\frac{1}{\lambda}$ representa la media de la distribución, expresada en minutos.

* **Distribución normal (método de Box-Müller):**

$$Z_0 = \sqrt{-2 \ln U_1} \cos(2\pi U_2), \quad U_1, U_2 \sim \text{Uniforme}(0, 1)$$

$$X = \mu + \sigma \cdot Z_0 \quad (\text{donde } \mu = 10, \; \sigma = 5)$$

### 4.2. Composición del tiempo de ocupación de pista

El tiempo total de servicio $T_{\text{servicio}}$ que ocupa un avión en una pista es la suma estocástica de cuatro componentes independientes:

$$T_{\text{servicio}} = T_{\text{combustible}} + T_{\text{maniobra}} + I_{\text{carga}} \cdot T_{\text{carga}} + I_{\text{rotura}} \cdot T_{\text{reparación}}$$

Donde:

* $T_{\text{combustible}} \sim \text{Exponencial}(\mu = 30 \text{ min})$; ocurre siempre.
* $T_{\text{maniobra}} \sim \text{Normal}(\mu = 10 \text{ min}, \sigma = 5 \text{ min})$; corresponde al aterrizaje y al despegue.
* $I_{\text{carga}} \sim \text{Bernoulli}(p = 0.5)$; $T_{\text{carga}} \sim \text{Exponencial}(\mu = 30 \text{ min})$.
* $I_{\text{rotura}} \sim \text{Bernoulli}(p = 0.1)$; $T_{\text{reparación}} \sim \text{Exponencial}(\mu = 15 \text{ min})$.

### 4.3. Estructura de eventos y estado del sistema

El motor de eventos discretos mantiene:

* **Reloj de simulación ($t$):** tiempo actual del sistema.
* **Lista de eventos futuros (LEF):** gestionada mediante `heapq`, almacenando tuplas del tipo `(tiempo, tipo_evento, pista_id)`.
* **Tipos de eventos:**
  * `ARRIBO`: llegada de un nuevo avión. Genera la fecha del próximo arribo si $t \le T$.
  * `SALIDA`: finalización de la ocupación de la pista por parte de un avión.
* **Variables de estado:** número de pistas libres (`pistas_libres`), contador de aviones en espera (`cola_espera`) y acumuladores del tiempo de inactividad por pista.

El estado del sistema incluye el número de pistas libres, el tamaño de la cola de espera y los acumuladores de tiempo inactivo para cada pista.



## Resultados de la Simulación

### Criterio de parada y réplicas independientes

Dado que una única simulación de una semana no es suficiente para obtener estimaciones fiables (debido a la aleatoriedad del proceso), hemos optado por el método de **réplicas independientes**. Agrupamos las ejecuciones en bloques de 10 y, tras cada bloque, calculamos la media y la desviación estándar de los tiempos de inactividad de cada pista.

El proceso se detiene cuando el error estándar máximo entre las 5 pistas es inferior a un umbral de precisión fijado en \( d = 10 \) minutos. Es decir, se cumple la condición:

\[
\max_{1 \le i \le 5} \frac{S_i}{\sqrt{n}} < 10
\]

donde \( S_i \) es la desviación estándar muestral de la pista i y \( n \) es el número total de réplicas realizadas.

### Resultados obtenidos

Tras realizar **1620 réplicas** independientes de una semana completa, hemos alcanzado el nivel de precisión deseado. Los resultados se resumen en la siguiente tabla:

| Pista | Tiempo inactivo medio (min) | Desviación estándar (min) | Error estándar (min) |
|-------|-----------------------------|---------------------------|----------------------|
| Pista 1 | 4408,8 | 399,7 | 9,93 |
| Pista 2 | 4414,9 | 396,1 | 9,84 |
| Pista 3 | 4413,9 | 401,3 | 9,97 |
| Pista 4 | 4430,3 | 400,5 | 9,95 |
| Pista 5 | 4413,6 | 399,3 | 9,92 |
| **Promedio** | **4416,3** | **399,4** | **9,92** |

### Análisis e interpretación de los resultados

- **Nivel de inactividad del aeropuerto:**  
  En promedio, cada pista permanece vacía **4416,3 minutos a la semana**, lo que equivale aproximadamente al **43,8% del tiempo total**. Esto implica que el aeropuerto opera con un nivel de ocupación cercano al 56,2%, lo que le permite cierto margen para absorber picos de demanda sin colapsar.

- **Equilibrio entre pistas:**  
  La diferencia máxima entre los tiempos inactivos medios de las distintas pistas es de solo **21,5 minutos**, lo que representa una variación inferior al 0,5%. Este resultado confirma que la asignación aleatoria uniforme de pistas funciona correctamente, distribuyendo la carga de manera equitativa y evitando que algunas pistas se sobrecarguen sistemáticamente.

- **Precisión y fiabilidad de las estimaciones:**  
  La desviación estándar observada (alrededor de 400 minutos) refleja la variabilidad natural del sistema de una semana a otra. Sin embargo, al haber realizado 1620 réplicas, hemos logrado reducir el error estándar por debajo de los 10 minutos en todas las pistas. Esto nos permite afirmar que nuestras estimaciones son robustas y que la incertidumbre asociada es pequeña, del orden de ±10 minutos como máximo.


El modelo desarrollado ofrece una visión clara y cuantitativa del comportamiento del aeropuerto, y las conclusiones obtenidas son sólidas desde el punto de vista estadístico. Esta herramienta puede ser de gran utilidad para la planificación de recursos y la gestión operativa en escenarios reales.